In [1]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

In [3]:
# 项目数据目录
data_path = "D:/adsb-flight-analysis/data"

# 读取航迹数据
df_trajectory = pd.read_csv(
    f"{data_path}/processed/trajectory.csv"
)

# 读取清洗后的飞行特征
flight_features_clean = pd.read_csv(
    f"{data_path}/processed/flight_features_clean.csv"
)

# 读取异常检测结果
flight_anomaly_results = pd.read_csv(
    f"{data_path}/processed/flight_anomaly_results_clean.csv"
)

# 读取聚类结果
flight_clustering_results = pd.read_csv(
    f"{data_path}/processed/flight_clustering_results_clean.csv"
)

print("所有数据读取成功！")
print("航迹数据：", df_trajectory.shape)
print("飞行特征：", flight_features_clean.shape)
print("异常检测：", flight_anomaly_results.shape)
print("聚类结果：", flight_clustering_results.shape)

所有数据读取成功！
航迹数据： (64692, 22)
飞行特征： (6689, 16)
异常检测： (6689, 18)
聚类结果： (6689, 19)


In [5]:
#全球 ADS-B 航空器空间分布图
import plotly.express as px
import plotly.io as pio
import os

# 使用浏览器显示 Plotly
pio.renderers.default = "browser"

# 创建结果文件夹
os.makedirs("D:/adsb-flight-analysis/results", exist_ok=True)

# 抽样，避免地图数据过多
df_map = df_trajectory.sample(
    n=min(10000, len(df_trajectory)),
    random_state=42
)

# 绘制全球航空器分布图
fig1 = px.scatter_geo(
    df_map,
    lat="latitude",
    lon="longitude",
    color="baro_altitude",
    hover_name="icao24",
    color_continuous_scale="Turbo",
    title="全球 ADS-B 航空器空间分布图",
    projection="natural earth"
)

fig1.update_layout(height=600)

# 保存 HTML
fig1.write_html(
    "D:/adsb-flight-analysis/results/aircraft_distribution.html"
)

# 在浏览器中打开
fig1.show()

In [8]:
#航空器平均飞行速度分布
import plotly.express as px
import plotly.io as pio
import os

pio.renderers.default = "browser"

# 只用于绘图：排除极端值
speed_data = flight_features_clean[
    flight_features_clean["avg_speed_kmh"].between(0, 1500)
].copy()

fig2 = px.histogram(
    speed_data,
    x="avg_speed_kmh",
    nbins=50,
    title="航空器平均飞行速度分布",
    labels={
        "avg_speed_kmh": "平均速度（km/h）",
        "count": "航空器数量"
    }
)

fig2.update_traces(
    hovertemplate=(
        "平均速度：%{x:.1f} km/h"
        "<br>航空器数量：%{y}"
        "<extra></extra>"
    )
)

fig2.update_layout(
    width=1100,
    height=600,

    title=dict(
        text="航空器平均飞行速度分布",
        x=0.5,
        xanchor="center",
        font=dict(size=24)
    ),

    xaxis=dict(
        title="平均速度（km/h）",
        range=[0, 1500],
        dtick=100,
        showgrid=True
    ),

    yaxis=dict(
        title="航空器数量",
        showgrid=True
    ),

    bargap=0.05,

    plot_bgcolor="white",
    paper_bgcolor="white",

    margin=dict(
        l=80,
        r=40,
        t=90,
        b=80
    )
)

# 保存
os.makedirs(
    "D:/adsb-flight-analysis/results",
    exist_ok=True
)

fig2.write_html(
    "D:/adsb-flight-analysis/results/speed_distribution.html"
)

fig2.show()

In [20]:
#异常检测结果图

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os

pio.renderers.default = "browser"

plot_data = flight_anomaly_results.copy()

# 删除异常分数或标签缺失的数据
plot_data = plot_data.dropna(
    subset=["anomaly_label", "anomaly_score"]
)

# 添加类别名称
plot_data["类别"] = plot_data["anomaly_label"].map({
    1: "正常航空器",
    -1: "异常航空器"
})

normal = plot_data[plot_data["anomaly_label"] == 1]
abnormal = plot_data[plot_data["anomaly_label"] == -1]

normal_count = len(normal)
abnormal_count = len(abnormal)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "① Isolation Forest 异常分数分布",
        "② 正常与异常特征对比",
        "③ Top 20 异常航空器"
    ],
    horizontal_spacing=0.12,
    column_widths=[0.34, 0.33, 0.33]
)

# ==============================
# 第一部分：异常分数分布
# ==============================

fig.add_trace(
    go.Histogram(
        x=normal["anomaly_score"],
        name="正常航空器",
        marker_color="rgba(74,144,226,0.65)",
        opacity=0.75,
        nbinsx=40,
        hovertemplate="异常分数：%{x:.4f}<br>数量：%{y}<extra>正常航空器</extra>"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=abnormal["anomaly_score"],
        name="异常航空器",
        marker_color="rgba(229,57,53,0.75)",
        opacity=0.8,
        nbinsx=40,
        hovertemplate="异常分数：%{x:.4f}<br>数量：%{y}<extra>异常航空器</extra>"
    ),
    row=1, col=1
)

feature_columns = [
    "distance_km",
    "avg_speed_kmh",
    "max_altitude_m",
    "max_vertical_rate",
    "total_heading_change"
]

feature_names = {
    "distance_km": "飞行距离",
    "avg_speed_kmh": "平均速度",
    "max_altitude_m": "最高高度",
    "max_vertical_rate": "最大垂直速度",
    "total_heading_change": "总航向变化"
}

feature_columns = [
    col for col in feature_columns
    if col in plot_data.columns
]

normal_values = []
abnormal_values = []
feature_labels = []

# 使用整体均值和标准差进行标准化
for col in feature_columns:

    values = pd.to_numeric(plot_data[col], errors="coerce")

    mean_value = values.mean()
    std_value = values.std()

    if pd.isna(std_value) or std_value == 0:
        continue

    normal_median = normal[col].median()
    abnormal_median = abnormal[col].median()

    if pd.isna(normal_median) or pd.isna(abnormal_median):
        continue

    normal_values.append(
        (normal_median - mean_value) / std_value
    )

    abnormal_values.append(
        (abnormal_median - mean_value) / std_value
    )

    feature_labels.append(feature_names[col])

# 正常航空器特征
fig.add_trace(
    go.Bar(
        x=feature_labels,
        y=normal_values,
        name="正常航空器特征",
        marker_color="#4A90E2",
        text=[f"{v:.2f}" for v in normal_values],
        textposition="outside",
        hovertemplate="%{x}<br>标准化中位数：%{y:.2f}<extra>正常航空器</extra>"
    ),
    row=1, col=2
)

# 异常航空器特征
fig.add_trace(
    go.Bar(
        x=feature_labels,
        y=abnormal_values,
        name="异常航空器特征",
        marker_color="#E53935",
        text=[f"{v:.2f}" for v in abnormal_values],
        textposition="outside",
        hovertemplate="%{x}<br>标准化中位数：%{y:.2f}<extra>异常航空器</extra>"
    ),
    row=1, col=2
)



top20 = (
    abnormal
    .sort_values("anomaly_score", ascending=True)
    .head(20)
    .sort_values("anomaly_score", ascending=True)
    .copy()
)

top20["icao24"] = top20["icao24"].astype(str)

fig.add_trace(
    go.Bar(
        x=top20["anomaly_score"],
        y=top20["icao24"],
        orientation="h",
        name="异常航空器排名",
        marker_color="#E53935",
        text=top20["anomaly_score"].map(lambda x: f"{x:.4f}"),
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "ICAO24：%{y}<br>"
            "异常分数：%{x:.5f}"
            "<extra>异常航空器</extra>"
        ),
        showlegend=False
    ),
    row=1, col=3
)

fig.update_layout(
    title=dict(
        text="ADS-B 航迹异常检测综合分析（Isolation Forest）",
        x=0.5,
        xanchor="center",
        font=dict(size=24)
    ),

    width=1650,
    height=850,

    template="plotly_white",

    barmode="overlay",

    hovermode="closest",

    legend=dict(
        orientation="h",
        y=1.08,
        x=0.5,
        xanchor="center"
    ),

    margin=dict(
        l=100,
        r=100,
        t=160,
        b=130
    ),

    font=dict(
        family="Arial",
        size=12
    ),

    annotations=[
        dict(
            text=f"正常航空器：{normal_count:,} 架",
            x=0.02,
            y=0.98,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(color="#4A90E2", size=13)
        ),
        dict(
            text=f"异常航空器：{abnormal_count:,} 架",
            x=0.02,
            y=0.93,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(color="#E53935", size=13)
        )
    ]
)



fig.update_xaxes(
    title_text="异常分数（越低越异常）",
    row=1, col=1
)

fig.update_yaxes(
    title_text="航空器数量",
    row=1, col=1
)

fig.update_xaxes(
    title_text="飞行特征",
    tickangle=-25,
    row=1, col=2
)

fig.update_yaxes(
    title_text="标准化中位数",
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor="gray",
    row=1, col=2
)

fig.update_xaxes(
    title_text="异常分数（越低越异常）",
    row=1, col=3
)

fig.update_yaxes(
    title_text="ICAO24 航空器编号",
    autorange="reversed",
    row=1, col=3
)



os.makedirs(
    "D:/adsb-flight-analysis/results",
    exist_ok=True
)

fig.write_html(
    "D:/adsb-flight-analysis/results/anomaly_detection.html"
)

fig.show()


In [22]:
#DBSCAN 航迹聚类结果可视化

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os

pio.renderers.default = "browser"

# 读取 DBSCAN 聚类结果
cluster_data = flight_clustering_results.copy()

# 聚类标签转换
cluster_data["cluster_type"] = cluster_data["cluster"].apply(
    lambda x: "噪声点" if x == -1 else f"Cluster {int(x)}"
)

# 统计各聚类数量
cluster_counts = (
    cluster_data["cluster_type"]
    .value_counts()
    .rename_axis("聚类类别")
    .reset_index(name="航空器数量")
)

# 计算各聚类平均特征
cluster_summary = (
    cluster_data[cluster_data["cluster"] != -1]
    .groupby("cluster_type")
    .agg(
        平均飞行距离=("distance_km", "mean"),
        平均飞行速度=("avg_speed_kmh", "mean")
    )
    .reset_index()
)

# 创建三个子图
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "各聚类航空器数量",
        "各聚类平均飞行距离",
        "各聚类平均飞行速度"
    ),
    specs=[
        [{"type": "bar"}, {"type": "bar"}, {"type": "bar"}]
    ]
)

# 第一张：聚类数量
fig.add_trace(
    go.Bar(
        x=cluster_counts["聚类类别"],
        y=cluster_counts["航空器数量"],
        name="航空器数量",
        marker_color="#4C78A8",
        text=cluster_counts["航空器数量"],
        texttemplate="%{text:,}",
        textposition="outside",
        cliponaxis=False,
        showlegend=False
    ),
    row=1, col=1
)

# 第二张：平均飞行距离
fig.add_trace(
    go.Bar(
        x=cluster_summary["cluster_type"],
        y=cluster_summary["平均飞行距离"],
        marker_color="#59A14F",
        name="平均距离",
        text=cluster_summary["平均飞行距离"],
        texttemplate="%{text:.2f} km",
        textposition="outside",
        cliponaxis=False,
        showlegend=False
    ),
    row=1, col=2
)

# 第三张：平均飞行速度
fig.add_trace(
    go.Bar(
        x=cluster_summary["cluster_type"],
        y=cluster_summary["平均飞行速度"],
        marker_color="#F28E2B",
        name="平均速度",
        text=cluster_summary["平均飞行速度"],
        texttemplate="%{text:.2f} km/h",
        textposition="outside",
        cliponaxis=False,
        showlegend=False
    ),
    row=1, col=3
)


fig.update_layout(
    title=dict(
        text="DBSCAN 航迹聚类综合分析",
        x=0.5,
        xanchor="center",
        font=dict(size=25)
    ),
    width=1450,
    height=700,
    template="plotly_white",
    margin=dict(
        l=70,
        r=50,
        t=150,
        b=100
    ),
    uniformtext=dict(
        minsize=10,
        mode="show"
    ),
    font=dict(
        family="Arial",
        size=12
    )
)


fig.update_yaxes(
    title_text="航空器数量",
    rangemode="tozero",
    row=1, col=1
)

fig.update_yaxes(
    title_text="平均飞行距离（km）",
    rangemode="tozero",
    row=1, col=2
)

fig.update_yaxes(
    title_text="平均飞行速度（km/h）",
    rangemode="tozero",
    row=1, col=3
)

os.makedirs(
    "D:/adsb-flight-analysis/results",
    exist_ok=True
)

fig.write_html(
    "D:/adsb-flight-analysis/results/dbscan_clustering.html"
)

fig.show()
